# ¿Se ven y se oyen iguales? · Juego multimodal
**Proyecto final · Módulo 6 · V1**

Cinco rondas para relacionar fotografías y sonidos de perro, gato, ave, automóvil y tren.
CLIP analiza los píxeles y CLAP analiza las ondas de audio. El jugador responde antes de conocer sus predicciones.

**Cómo ejecutar:** en Google Colab selecciona *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU* (opcional pero recomendado), y después *Ejecutar todas*. La primera ejecución descarga dependencias, 30 recursos y modelos públicos; necesita Internet. CPU también funciona, con mayor espera. Al final aparece el juego y un enlace temporal de Gradio que depende de mantener abierto el runtime.

**Datos:** [repositorio público](https://github.com/hectorzid23-cmyk/juego-multimodal). Se usa una revisión fija y se comprueban hashes. No es necesario montar Drive, iniciar sesión en Hugging Face ni proporcionar claves.
**Alcance:** un modelo de visión + uno de audio + Gradio, según PROJECT_BRIEF_TRIM. GitHub sustituye Drive para los recursos por decisión del autor. No se entrena con estos 30 ejemplos.

La puntuación se calcula con las categorías revisadas del dataset. Las predicciones de la IA se muestran por separado: **un error del modelo nunca resta puntos al niño**. Las etiquetas de carpeta no entran en los modelos.


In [1]:
import subprocess, sys, importlib.metadata as metadata
PACKAGES = [
    "torch==2.7.1", "torchvision==0.22.1", "transformers==4.51.3",
    "gradio==5.49.1", "librosa==0.11.0", "soundfile==0.13.1",
    "numpy==2.2.6", "pillow==11.3.0", "pandas==2.2.3",
]
# Ejecutar antes de importar las bibliotecas, en un runtime nuevo.
missing = []
for requirement in PACKAGES:
    name, version = requirement.split("==")
    try:
        if metadata.version(name) != version:
            missing.append(requirement)
    except metadata.PackageNotFoundError:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Dependencias preparadas. Python:", sys.version.split()[0])


Dependencias preparadas. Python: 3.12.12


## 1. Entorno reproducible
La semilla fija hace reproducibles las pruebas. Cada nueva partida recibe una semilla propia para variar las rondas. La inferencia se realiza en modo evaluación, sin gradientes.

In [2]:
import os, random, time, json, hashlib, io, zipfile, urllib.request
from pathlib import Path
from copy import deepcopy
from threading import RLock
import numpy as np
import pandas as pd
import torch
import soundfile as sf
import librosa
from PIL import Image, ImageOps
from IPython.display import display
import gradio as gr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
else:
    torch.set_num_threads(min(4, os.cpu_count() or 1))
BASE = Path("/content") if Path("/content").exists() else Path.cwd()
RUNTIME = BASE / "juego_runtime"
RUNTIME.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(RUNTIME / "models")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
print("Dispositivo:", DEVICE)
print({r.split("==")[0]: metadata.version(r.split("==")[0]) for r in PACKAGES})


Dispositivo: cpu
{'torch': '2.7.1', 'torchvision': '0.22.1', 'transformers': '4.51.3', 'gradio': '5.49.1', 'librosa': '0.11.0', 'soundfile': '0.13.1', 'numpy': '2.2.6', 'pillow': '11.3.0', 'pandas': '2.2.3'}


## 2. Dataset: descarga y validación
Se conservan los recursos originales. Las carpetas proporcionan la respuesta de referencia para puntuar y evaluar; las funciones de inferencia solo reciben una imagen o una señal de audio.

In [3]:
DATA_COMMIT = "c83c5248b672ffdd52ff6e18cb0e8bebc11e2842"
REPO = "hectorzid23-cmyk/juego-multimodal"
DATA_ROOT = RUNTIME / ("dataset-" + DATA_COMMIT[:12])
MANIFEST = [
  {
    "path": "datos/automovil/audios/automovil_01.wav",
    "sha256": "2911cf60a73a2b247113e2a15b302a048a324892310b9137a5f977d25bece35f"
  },
  {
    "path": "datos/automovil/audios/automovil_02.wav",
    "sha256": "3b5c4e08bcfa77fab0b18bbd17d3a3390525c2c77114d2b86ce0b3011206f96e"
  },
  {
    "path": "datos/automovil/audios/automovil_03.wav",
    "sha256": "79a3dbf3e7a4804cd95e5bb3311a196f074b53a9c32d07a1cbb90f2f5720c6a9"
  },
  {
    "path": "datos/automovil/imagenes/automovil_01.jpg",
    "sha256": "ccc61451849e4c327ca364f9dcf69ce34b91fea6d8577d6f82b63f3be62e318f"
  },
  {
    "path": "datos/automovil/imagenes/automovil_02.jpg",
    "sha256": "19f695740d82fcb3127edbd938c3adced1af6e08b1f82afe3eaa35d32e8d1a15"
  },
  {
    "path": "datos/automovil/imagenes/automovil_03.jpg",
    "sha256": "2db5f477b6b2222be6f2c1a4628d571e83f84826627529e39ff689ec4a9b2620"
  },
  {
    "path": "datos/ave/audios/ave_01.wav",
    "sha256": "471c3a51b3d28bd086a33c54868e231f929e5cc5a2d0b0e33e7cccc776c88a37"
  },
  {
    "path": "datos/ave/audios/ave_02.wav",
    "sha256": "4e98e9c8aa2a84b50c60dd27488bcb049cbcc6619642c0c713f9e1b4867ed7b5"
  },
  {
    "path": "datos/ave/audios/ave_03.wav",
    "sha256": "ca2be8018f8c15583d9da14d10c7010a066aa42a043082247bbb9eb4d7b84b05"
  },
  {
    "path": "datos/ave/imagenes/ave_01.jpg",
    "sha256": "d52048e3820fabb1c69ceabb38e5e05a9a06465889c0ed279ba4bd91825a5ca9"
  },
  {
    "path": "datos/ave/imagenes/ave_02.jpg",
    "sha256": "3f99eecb644a34327f264d0852e203d8b1edd502c9790fb4a2d82a887e113577"
  },
  {
    "path": "datos/ave/imagenes/ave_03.jpg",
    "sha256": "d9cbaa4bc5ead34195a933b67986c022cdb41fe399aadcc7ebd7a72e98b87e4a"
  },
  {
    "path": "datos/gato/audios/gato_01.wav",
    "sha256": "33d7df9375b6ab9db42d0c91130186ff6374c504a7bd4b1b094d461461ff76d3"
  },
  {
    "path": "datos/gato/audios/gato_02.wav",
    "sha256": "be7e6cbbff1f22701397cdb22c69502439864028100a2ffcfe594fd067fb4d07"
  },
  {
    "path": "datos/gato/audios/gato_03.wav",
    "sha256": "b59198c495da718dc0715f1dd433e14e7b70a4b0cd8dd2755735d3938a2aad51"
  },
  {
    "path": "datos/gato/imagenes/gato_01.jpg",
    "sha256": "ec8ff0a6954a5f54e4ad246f0a788d076f675e457bca16a04248718cca3fe631"
  },
  {
    "path": "datos/gato/imagenes/gato_02.jpg",
    "sha256": "59ec9bf12a5b63e0913e986b9566b96228c9f8921fda4fb87bf2a7f9acff3dd2"
  },
  {
    "path": "datos/gato/imagenes/gato_03.jpg",
    "sha256": "e9b6d15f2bb68da459c9cbf93a5bcb260800753afc0d7df39df3304cb16d2fd3"
  },
  {
    "path": "datos/perro/audios/perro_01.wav",
    "sha256": "ab5381cf9a77f6b530b48c4f6622d94357f736b676e9cd2a64e13c55e629be8a"
  },
  {
    "path": "datos/perro/audios/perro_02.wav",
    "sha256": "27ec7f239415913309ceceeb5845b8925ded528eb6255cf3b085df189b102dd1"
  },
  {
    "path": "datos/perro/audios/perro_03.wav",
    "sha256": "ffb848081ea9c5516758c32725c4a3acd6b7369c3cdfa52192ac0d7005519569"
  },
  {
    "path": "datos/perro/imagenes/perro_01.jpg",
    "sha256": "e75408954a87f0a56e2a3f97e4c47c7e451a6e5da7d92c58b3519c3f9261b7f2"
  },
  {
    "path": "datos/perro/imagenes/perro_02.jpg",
    "sha256": "5c3161b36551c8042cbee1ce4462f2cdd7afe2604dca148986aea9863d71dafa"
  },
  {
    "path": "datos/perro/imagenes/perro_03.jpg",
    "sha256": "58915a7b33280571b43419c7aef24e8f3b1059e95d0414918951a2c043ef7874"
  },
  {
    "path": "datos/tren/audios/tren_01.wav",
    "sha256": "cf563dbf235c7d0c349fdcd41c5027833d73fe8cad236aa729d07f5cc870c080"
  },
  {
    "path": "datos/tren/audios/tren_02.wav",
    "sha256": "92edb1265ec02d7f8e3ee7c2dee687150de1804d0cbfbcbb1dde0d771c8640e6"
  },
  {
    "path": "datos/tren/audios/tren_03.wav",
    "sha256": "dcf6b90a67308a2b920b924f14e0bd34b4ac63622ddc66c38bf9ec019712d3ef"
  },
  {
    "path": "datos/tren/imagenes/tren_01.jpg",
    "sha256": "0deb551015e4e0ccdecbeee196f21466388d3391ce875e139ad736d06bd5d1f1"
  },
  {
    "path": "datos/tren/imagenes/tren_02.jpg",
    "sha256": "1b6f60415f340accc6e447d37df1b14ef5d764e2df18bf29292e3bc9d0a25ccc"
  },
  {
    "path": "datos/tren/imagenes/tren_03.jpg",
    "sha256": "ed2e8d9e1a679e329d0c36ea8608bfade33cb4196139fddb18713ca8de9519f0"
  }
]
CATEGORIES = ["perro", "gato", "ave", "automovil", "tren"]
NAMES = dict(zip(CATEGORIES, ["Perro", "Gato", "Ave", "Automóvil", "Tren"]))

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def download_dataset():
    valid = all((DATA_ROOT / e["path"]).is_file() and
                sha256(DATA_ROOT / e["path"]) == e["sha256"] for e in MANIFEST)
    if valid:
        return
    url = f"https://codeload.github.com/{REPO}/zip/{DATA_COMMIT}"
    with urllib.request.urlopen(url, timeout=120) as response:
        archive = response.read()
    with zipfile.ZipFile(io.BytesIO(archive)) as z:
        # Solo extraer los 30 nombres esperados, sin extraer rutas arbitrarias.
        prefix = f"juego-multimodal-{DATA_COMMIT}/"
        for e in MANIFEST:
            content = z.read(prefix + e["path"])
            if hashlib.sha256(content).hexdigest() != e["sha256"]:
                raise ValueError("El dataset no coincide con la revisión fijada.")
            dest = DATA_ROOT / e["path"]
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_bytes(content)

download_dataset()
ASSETS = []
for e in MANIFEST:
    relative = Path(e["path"])
    category = relative.parts[1]
    kind = "image" if relative.suffix == ".jpg" else "audio"
    path = DATA_ROOT / relative
    if kind == "image":
        with Image.open(path) as im:
            im.verify()
    else:
        info = sf.info(path)
        assert info.frames > 0 and info.samplerate > 0, path
    ASSETS.append(dict(path=str(path), category=category, kind=kind))
for category in CATEGORIES:
    for kind in ["image", "audio"]:
        assert sum(a["category"] == category and a["kind"] == kind for a in ASSETS) == 3
print("Dataset verificado: 15 fotografías y 15 audios, cinco categorías.")
display(pd.DataFrame(ASSETS).groupby(["category", "kind"]).size().unstack())


Dataset verificado: 15 fotografías y 15 audios, cinco categorías.
kind       audio  image
category               
automovil      3      3
ave            3      3
gato           3      3
perro          3      3
tren           3      3


## 3. Modelos y preprocesamiento
Se usan **CLIP ViT-B/32** (visión) y **CLAP HTSAT unfused** (sonido), de los materiales L24/L25. Se comparan cinco descripciones adecuadas a cada modalidad y se unen sus resultados por categoría. Los espacios de embeddings de CLIP y CLAP no se comparan directamente.

La puntuación de cada modelo es relativa a estas cinco opciones. Un 90 % no significa una probabilidad calibrada del 90 % de tener razón. Fuera de estas categorías puede producirse una respuesta segura e incorrecta; esta V1 utiliza únicamente el dataset cerrado.

Referencias: [CLAP](https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/clap), [CLIP](https://huggingface.co/openai/clip-vit-base-patch32), [CLAP checkpoint](https://huggingface.co/laion/clap-htsat-unfused).


In [4]:
from transformers import CLIPModel, CLIPProcessor, ClapModel, ClapProcessor
CLIP_ID = "openai/clip-vit-base-patch32"
CLAP_ID = "laion/clap-htsat-unfused"
CLIP_REV = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
CLAP_REV = "8fa0f1c6d0433df6e97c127f64b2a1d6c0dcda8a"
CACHE = str(RUNTIME / "models")
# Revisiones fijas: mismos pesos y procesadores en cada ejecución.
from huggingface_hub import snapshot_download
MODEL_FILES = ["config.json", "preprocessor_config.json", "tokenizer_config.json",
               "special_tokens_map.json", "tokenizer.json", "vocab.json",
               "merges.txt", "pytorch_model.bin"]
print("Descargando/cargando CLIP...", flush=True)
clip_path = snapshot_download(CLIP_ID, revision=CLIP_REV, cache_dir=CACHE,
                              allow_patterns=MODEL_FILES, max_workers=2)
clip_processor = CLIPProcessor.from_pretrained(clip_path, local_files_only=True, use_fast=False)
clip_model = CLIPModel.from_pretrained(clip_path, local_files_only=True, use_safetensors=False).to(DEVICE).eval()
print("Descargando/cargando CLAP...", flush=True)
clap_path = snapshot_download(CLAP_ID, revision=CLAP_REV, cache_dir=CACHE,
                              allow_patterns=MODEL_FILES, max_workers=2)
clap_processor = ClapProcessor.from_pretrained(clap_path, local_files_only=True)
clap_model = ClapModel.from_pretrained(clap_path, local_files_only=True, use_safetensors=False).to(DEVICE).eval()
print("CLIP:", getattr(clip_model.config, "_commit_hash", None))
print("CLAP:", getattr(clap_model.config, "_commit_hash", None))
print("Parámetros:", {name: sum(p.numel() for p in model.parameters())
                       for name, model in [("CLIP", clip_model), ("CLAP", clap_model)]})
if DEVICE == "cuda":
    print("VRAM reservada (GB):", round(torch.cuda.memory_reserved() / 1e9, 2))

IMAGE_TEXTS = [
    "a photo of a dog", "a photo of a cat", "a photo of a bird",
    "a photo of a car", "a photo of a train",
]
AUDIO_TEXTS = [
    "the sound of a dog barking", "the sound of a cat meowing",
    "the sound of birds chirping", "the sound of a car engine driving",
    "the sound of a train moving on railway tracks",
]
MODEL_LOCK = RLock()

def prepare_image(image):
    if not isinstance(image, Image.Image):
        raise ValueError("Se necesita una fotografía válida.")
    if min(image.size) < 32:
        raise ValueError("La imagen es demasiado pequeña.")
    image = ImageOps.exif_transpose(image).convert("RGB")
    image.thumbnail((1024, 1024))
    return image

def prepare_audio(waveform, sr):
    if not isinstance(sr, (int, np.integer)) or sr <= 0:
        raise ValueError("Frecuencia de muestreo inválida.")
    x = np.asarray(waveform)
    if x.ndim not in (1, 2) or x.size == 0:
        raise ValueError("El audio está vacío o tiene una forma inválida.")
    if x.dtype.kind in "iu":
        info = np.iinfo(x.dtype)
        midpoint = (info.max + 1) / 2 if x.dtype.kind == "u" else 0
        scale = midpoint if midpoint else max(abs(info.min), info.max)
        x = (x.astype(np.float32) - midpoint) / scale
    else:
        x = x.astype(np.float32)
    if x.ndim == 2:  # soundfile: muestras x canales
        if x.shape[1] > 8:
            raise ValueError("Formato de canales no admitido.")
        x = x.mean(axis=1)
    if not np.isfinite(x).all():
        raise ValueError("El audio contiene valores inválidos.")
    if not 0.5 <= len(x) / sr <= 30:
        raise ValueError("El audio debe durar entre 0.5 y 30 segundos.")
    peak = float(np.max(np.abs(x)))
    if peak < 1e-5:
        raise ValueError("El audio está en silencio.")
    if peak > 1:
        x = x / peak
    if sr != 48000:
        target_length = round(len(x) * 48000 / sr)
        x = librosa.resample(x, orig_sr=sr, target_sr=48000)
        x = librosa.util.fix_length(x, size=target_length)
    return np.ascontiguousarray(x, dtype=np.float32)

def prediction(logits, started):
    scores = torch.softmax(logits.float(), dim=-1).detach().cpu().numpy().reshape(-1)
    index = int(scores.argmax())
    return {"category": CATEGORIES[index], "score": float(scores[index]),
            "scores": dict(zip(CATEGORIES, map(float, scores))),
            "seconds": time.perf_counter() - started}

def infer_image(image):
    image = prepare_image(image)
    started = time.perf_counter()
    with MODEL_LOCK, torch.inference_mode():
        batch = clip_processor(text=IMAGE_TEXTS, images=image, return_tensors="pt", padding=True)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        return prediction(clip_model(**batch).logits_per_image[0], started)

def infer_audio(waveform, sr):
    audio = prepare_audio(waveform, sr)
    started = time.perf_counter()
    with MODEL_LOCK, torch.inference_mode():
        batch = clap_processor(text=AUDIO_TEXTS, audios=audio, sampling_rate=48000,
                               return_tensors="pt", padding=True)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        return prediction(clap_model(**batch).logits_per_audio[0], started)

def infer_pair(pair):
    with Image.open(pair["image"]["path"]) as im:
        visual = infer_image(im)
    waveform, sr = sf.read(pair["audio"]["path"], dtype="float32")
    audio = infer_audio(waveform, sr)
    return visual, audio


Descargando/cargando CLIP...
Descargando/cargando CLAP...
CLIP: 3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268
CLAP: 8fa0f1c6d0433df6e97c127f64b2a1d6c0dcda8a
Parámetros: {'CLIP': 151277313, 'CLAP': 153492890}


## 4. Motor del juego
Las categorías de referencia se utilizan aquí para puntuar. Los modelos vuelven a procesar los recursos al responder cada ronda; la evaluación previa no sustituye la inferencia del juego.

In [5]:
def make_game(seed=None):
    if seed is None:
        seed = random.SystemRandom().randrange(2**32)
    rng = random.Random(seed)
    matches = rng.choice([2, 3])
    images = [a for a in ASSETS if a["kind"] == "image"]
    audios = [a for a in ASSETS if a["kind"] == "audio"]
    same = [{"image": i, "audio": a} for i in images for a in audios
            if i["category"] == a["category"]]
    different = [{"image": i, "audio": a} for i in images for a in audios
                 if i["category"] != a["category"]]
    rounds = rng.sample(same, matches) + rng.sample(different, 5 - matches)
    rng.shuffle(rounds)
    return {"seed": seed, "rounds": rounds, "index": 0, "points": 0,
            "answered": False, "finished": False, "history": []}

def truth(pair):
    return pair["image"]["category"] == pair["audio"]["category"]

def answer_game(state, choice, infer=infer_pair):
    if not state or state["finished"] or state["answered"]:
        return state, None  # idempotencia: no ejecutar ni sumar otra vez
    pair = state["rounds"][state["index"]]
    visual, audio = infer(pair)  # si falla, no modificar estado ni puntos
    correct = bool(choice) == truth(pair)
    result = {"correct": correct, "real_match": truth(pair), "visual": visual,
              "audio": audio, "ai_match": visual["category"] == audio["category"],
              "image_truth": pair["image"]["category"], "audio_truth": pair["audio"]["category"]}
    updated = deepcopy(state)
    updated["points"] += int(correct)
    updated["answered"] = True
    updated["history"].append(result)
    return updated, result

def next_round(state):
    if not state or not state["answered"] or state["finished"]:
        return state
    updated = deepcopy(state)
    if updated["index"] == 4:
        updated["finished"] = True
    else:
        updated["index"] += 1
        updated["answered"] = False
    return updated


## 5. Pruebas del juego y de entradas
Las predicciones simuladas en esta celda prueban únicamente las reglas de puntuación. La siguiente celda realiza la evaluación con modelos reales.

In [6]:
def run_rule_tests():
    def fake(pair):
        # Error deliberado: ambos modelos dicen perro, sea cual sea la entrada.
        p = {"category": "perro", "score": 0.9, "scores": {}, "seconds": 0}
        return deepcopy(p), deepcopy(p)
    for seed in range(30):
        g = make_game(seed)
        pairs = [(p["image"]["path"], p["audio"]["path"]) for p in g["rounds"]]
        assert len(set(pairs)) == 5
        assert sum(map(truth, g["rounds"])) in (2, 3)
        assert next_round(g) == g
        for n in range(5):
            before = deepcopy(g)
            g, result = answer_game(g, truth(g["rounds"][n]), infer=fake)
            assert g["points"] == n + 1
            assert before["answered"] is False
            repeated, _ = answer_game(g, True, infer=fake)
            assert repeated == g
            g = next_round(g)
        assert g["finished"] and g["points"] == 5
    g = make_game(7)
    other = make_game(8)
    saved = deepcopy(g)
    def fails(pair):
        raise RuntimeError("fallo de prueba")
    try:
        answer_game(g, True, infer=fails)
        raise AssertionError("Debió fallar")
    except RuntimeError:
        pass
    assert g == saved and other["points"] == 0
    wrong, _ = answer_game(g, not truth(g["rounds"][0]), infer=fake)
    assert wrong["points"] == 0
    assert make_game(42) == make_game(42)
    for invalid in [np.zeros(48000), np.full(48000, np.nan), np.ones(10)]:
        try:
            prepare_audio(invalid, 48000)
            raise AssertionError("Debió rechazarse el audio")
        except ValueError:
            pass
    stereo = np.full((44100, 2), 1000, dtype=np.int16)
    assert prepare_audio(stereo, 44100).shape == (48000,)
    assert prepare_image(Image.new("L", (128, 128))).mode == "RGB"
    assert prepare_image(Image.new("RGBA", (128, 128))).mode == "RGB"
    print("Pruebas correctas: 30 partidas, idempotencia, errores, reinicio y entradas.")
run_rule_tests()


Pruebas correctas: 30 partidas, idempotencia, errores, reinicio y entradas.


## 6. Evaluación exploratoria real
Se procesan todos los archivos. Esta tabla describe solo estos ejemplos, no la precisión general de los modelos. Las confusiones se mantienen visibles. En CPU esta celda puede tardar varios minutos.

In [7]:
evaluation_rows = []
for number, asset in enumerate(ASSETS, 1):
    if asset["kind"] == "image":
        with Image.open(asset["path"]) as im:
            pred = infer_image(im)
    else:
        wave, sr = sf.read(asset["path"], dtype="float32")
        pred = infer_audio(wave, sr)
    evaluation_rows.append({
        "archivo": Path(asset["path"]).name, "modalidad": asset["kind"],
        "real": asset["category"], "prediccion": pred["category"],
        "acierto": pred["category"] == asset["category"],
        "puntuacion": round(pred["score"], 4), "segundos": round(pred["seconds"], 3),
    })
    print(f"{number}/30", Path(asset["path"]).name, pred["category"])
evaluation = pd.DataFrame(evaluation_rows)
display(evaluation)
display(evaluation.groupby("modalidad")["acierto"].agg(["sum", "count", "mean"]))
display(evaluation.loc[~evaluation["acierto"]])
# Evidencia descargable desde el panel Archivos del runtime.
evaluation.to_csv(RUNTIME / "evaluacion.csv", index=False)

# La inferencia recibe contenido en memoria, nunca nombres o categorías:
first_image = next(a for a in ASSETS if a["kind"] == "image")
with Image.open(first_image["path"]) as original:
    pixel_copy = Image.fromarray(np.asarray(original.convert("RGB")).copy())
    left, right = infer_image(original), infer_image(pixel_copy)
assert left["category"] == right["category"]
assert np.allclose(list(left["scores"].values()), list(right["scores"].values()), atol=1e-5)
print("Invariancia al nombre: píxeles idénticos producen la misma predicción.")
first_audio = next(a for a in ASSETS if a["kind"] == "audio")
wave, sr = sf.read(first_audio["path"], dtype="float32")
left, right = infer_audio(wave, sr), infer_audio(wave.copy(), sr)
assert left["category"] == right["category"]
assert np.allclose(list(left["scores"].values()), list(right["scores"].values()), atol=1e-5)
print("Invariancia al nombre: señales idénticas producen la misma predicción.")


1/30 automovil_01.wav automovil
2/30 automovil_02.wav gato
3/30 automovil_03.wav automovil
4/30 automovil_01.jpg automovil
5/30 automovil_02.jpg automovil
6/30 automovil_03.jpg automovil
7/30 ave_01.wav ave
8/30 ave_02.wav ave
9/30 ave_03.wav ave
10/30 ave_01.jpg ave
11/30 ave_02.jpg ave
12/30 ave_03.jpg ave
13/30 gato_01.wav gato
14/30 gato_02.wav gato
15/30 gato_03.wav gato
16/30 gato_01.jpg gato
17/30 gato_02.jpg gato
18/30 gato_03.jpg gato
19/30 perro_01.wav perro
20/30 perro_02.wav perro
21/30 perro_03.wav perro
22/30 perro_01.jpg perro
23/30 perro_02.jpg perro
24/30 perro_03.jpg perro
25/30 tren_01.wav tren
26/30 tren_02.wav tren
27/30 tren_03.wav tren
28/30 tren_01.jpg tren
29/30 tren_02.jpg tren
30/30 tren_03.jpg tren
             archivo modalidad       real  ... acierto  puntuacion  segundos
0   automovil_01.wav     audio  automovil  ...    True      0.8622     0.491
1   automovil_02.wav     audio  automovil  ...   False      0.4585     0.343
2   automovil_03.wav     audio  a

## 7. Aplicación
Pulsa **Comenzar partida**, escucha el audio y decide. Tras responder se muestran la solución y el análisis de IA. El siguiente paso siempre requiere pulsar un botón. Cada visitante tiene su propia partida; las operaciones se serializan para evitar respuestas dobles simultáneas.

In [8]:
def render_round(state):
    if not state:
        return "Pulsa Comenzar partida.", None, None
    if state["finished"]:
        return f"### ¡Partida terminada!\nAcertaste **{state['points']} de 5**. ¡Gracias por jugar!", None, None
    pair = state["rounds"][state["index"]]
    with Image.open(pair["image"]["path"]) as im:
        picture = im.convert("RGB").copy()
    wave, sr = sf.read(pair["audio"]["path"], dtype="float32")
    title = f"### Ronda {state['index'] + 1} de 5 · Puntos: {state['points']}"
    # Entregar contenido, sin revelar los nombres de los archivos del dataset.
    return title, picture, (sr, wave)

def feedback(result):
    outcome = "¡Acertaste! +1 punto." if result["correct"] else "Esta vez no. ¡Sigue intentándolo!"
    image_name, audio_name = NAMES[result["image_truth"]], NAMES[result["audio_truth"]]
    real = "Sí coinciden" if result["real_match"] else "No coinciden"
    v, a = result["visual"], result["audio"]
    mistaken = v["category"] != result["image_truth"] or a["category"] != result["audio_truth"]
    warning = "**La IA se equivocó en alguna categoría. Tu puntuación usa la respuesta real.**\n\n" if mistaken else ""
    detail = (f"{warning}Fotografía: **{NAMES[v['category']]}** · puntuación relativa: **{v['score']:.1%}**\n\n"
              f"Sonido: **{NAMES[a['category']]}** · puntuación relativa: **{a['score']:.1%}**\n\n"
              f"Comparación de la IA: **{'Coinciden' if result['ai_match'] else 'No coinciden'}**.\n\n"
              "Estas puntuaciones comparan cinco opciones; no garantizan que la IA tenga razón.")
    return f"### {outcome}\n{real}: imagen de **{image_name}**, sonido de **{audio_name}**.", detail

def start_ui():
    g = make_game()
    title, picture, wave = render_round(g)
    return g, title, picture, wave, "", "", gr.update(interactive=True), gr.update(interactive=True), gr.update(interactive=False)

def respond_ui(state, choice, progress=gr.Progress()):
    if not state or state["answered"] or state["finished"]:
        return (state,) + (gr.skip(),) * 6
    try:
        progress(0.1, desc="La IA está mirando y escuchando...")
        updated, result = answer_game(state, choice)
        message, detail = feedback(result)
        progress(1, desc="¡Listo!")
        title = f"### Ronda {updated['index'] + 1} de 5 · Puntos: {updated['points']}"
        return updated, title, message, detail, gr.update(interactive=False), gr.update(interactive=False), gr.update(interactive=True)
    except Exception as error:
        print("Inferencia:", type(error).__name__, str(error))
        return (state, gr.skip(), "No se pudo analizar esta ronda. Vuelve a pulsar tu respuesta; tus puntos no cambiaron.",
                "", gr.update(interactive=True), gr.update(interactive=True), gr.update(interactive=False))

def yes_ui(state, progress=gr.Progress()):
    return respond_ui(state, True, progress)

def no_ui(state, progress=gr.Progress()):
    return respond_ui(state, False, progress)

def next_ui(state):
    updated = next_round(state)
    if updated == state:
        return (state,) + (gr.skip(),) * 8
    title, picture, wave = render_round(updated)
    active = not updated["finished"]
    return updated, title, picture, wave, "", "", gr.update(interactive=active), gr.update(interactive=active), gr.update(interactive=False)

# Cerrar únicamente una instancia previa de esta app si se vuelve a ejecutar la celda.
if "demo" in globals():
    demo.close()
with gr.Blocks(theme=gr.themes.Soft(), title="¿Se ven y se oyen iguales?") as demo:
    gr.Markdown("# ¿Se ven y se oyen iguales?\nMira, escucha y elige. ¡Tienes cinco rondas!")
    state = gr.State(value=None)
    start = gr.Button("Comenzar / Nueva partida", variant="primary")
    title = gr.Markdown("Pulsa Comenzar partida.")
    with gr.Row():
        picture = gr.Image(label="Mira la fotografía", type="pil", interactive=False, height=330,
                           show_download_button=False)
        wave = gr.Audio(label="Escucha el sonido · Puedes reproducirlo otra vez",
                        interactive=False, autoplay=False, show_download_button=False)
    with gr.Row():
        yes = gr.Button("Sí coinciden", variant="primary", interactive=False)
        no = gr.Button("No coinciden", interactive=False)
    message = gr.Markdown()
    with gr.Accordion("¿Qué encontró la inteligencia artificial?", open=True):
        detail = gr.Markdown()
    next_button = gr.Button("Siguiente ronda / Ver resultado", interactive=False)
    gr.Markdown("Puntos basados en la respuesta real. La IA puede equivocarse. No se solicitan nombres ni grabaciones personales.")
    common = [state, title, picture, wave, message, detail, yes, no, next_button]
    start.click(start_ui, outputs=common, concurrency_id="game", concurrency_limit=1)
    yes.click(yes_ui, inputs=state, outputs=[state, title, message, detail, yes, no, next_button],
              concurrency_id="game", concurrency_limit=1)
    no.click(no_ui, inputs=state, outputs=[state, title, message, detail, yes, no, next_button],
             concurrency_id="game", concurrency_limit=1)
    next_button.click(next_ui, inputs=state, outputs=common, concurrency_id="game", concurrency_limit=1)
demo.queue(max_size=16)
print("Interfaz construida.")


Interfaz construida.


In [9]:
# En Colab, share=True genera el enlace temporal para jugar.
# Fuera de Colab se sirve en localhost.
IN_COLAB = "google.colab" in sys.modules
demo.launch(share=IN_COLAB, inline=IN_COLAB, prevent_thread_lock=True,
            show_error=False, quiet=True)


## Limitaciones y referencias
- Solo cinco categorías y tres ejemplos por modalidad/categoría. No es un benchmark ni se promete reconocimiento de entradas arbitrarias.
- Los nombres de carpeta son referencias humanas y deben revisarse si se reemplaza el dataset.
- CLIP y CLAP pueden coincidir y estar equivocados. Se muestran sus errores sin penalizar al jugador.
- Los audios no se reproducen automáticamente. No se recopilan datos personales ni se habilitan cargas de menores.
- Si faltan red o memoria, revisar el mensaje de la celda. En Colab usar un runtime nuevo; la CPU ofrece una alternativa más lenta.
- El enlace Gradio deja de funcionar al terminar el runtime; GitHub conserva notebook y dataset.
- Datos y notebook: CC0 del repositorio. Los modelos y bibliotecas conservan sus propias licencias.

Fuentes: materiales locales L24/L25 y guía V1, [CLIP](https://huggingface.co/openai/clip-vit-base-patch32),
[CLAP](https://huggingface.co/laion/clap-htsat-unfused),
[Gradio State](https://www.gradio.app/guides/state-in-blocks).


## Validación de esta entrega

Ejecución real en un entorno virtual nuevo de **Windows, Python 3.12.12 y CPU**, con las versiones de la celda de instalación. Las nueve celdas de código se ejecutaron en orden y las salidas guardadas proceden de esa ejecución.

- CLIP: **15/15** imágenes clasificadas correctamente.
- CLAP: **14/15** audios clasificados correctamente.
- Error observado: `automovil_02.wav` fue clasificado como gato, con puntuación relativa 0.4585. No se cambió su etiqueta ni se ocultó el resultado. La causa precisa requiere revisar el contenido sonoro; la semejanza acústica y la formulación del texto candidato son hipótesis, no causas verificadas.
- Pasaron 30 partidas de prueba, protección frente a doble respuesta, entradas inválidas, error de inferencia sin pérdida de estado y puntuación independiente de la IA.
- La app Gradio se levantó localmente y se recorrió mediante su API HTTP: cinco rondas, intento de doble respuesta, final, reinicio y dos sesiones independientes.
- Se comprobó que los mismos píxeles y ondas, sin metadatos de nombres, mantienen las predicciones.

**Pendiente:** ensayo dentro de Google Colab/T4, enlace público y revisión visual con un usuario. No se presenta la prueba local como si hubiera ocurrido en Colab. Estos 30 recursos son una muestra de demostración, no una medida de rendimiento general.
